# Single-Cell Trajectory Inference

This notebook demonstrates OTP-FM for trajectory inference on the Embryoid Body (EB) single-cell RNA-seq dataset.

The EB dataset contains cells across 5 time points during embryoid body differentiation, providing a benchmark for single-cell trajectory inference methods.

In [ ]:
import torch
import numpy as np
from collections import OrderedDict
from pathlib import Path

# Import OTP-FM
from otpfm import OTPFM
from otpfm.potentials import W2InfPotential

# Import experiment utilities
from experiments.singlecell.data import load_eb_data, create_eb_dataloaders
from experiments.singlecell import EBTrainer
from experiments.singlecell.plotting import plot_pca_trajectories

## 1. Load Data

The EB data will be downloaded automatically if not present.

In [ ]:
# Load and preprocess data
data = load_eb_data(
    data_dir=Path("data/singlecell"),
    pca_dim=100,
    normalize=True,
    ot_coupling=True,  # Use OT-aligned samples for training
)

print(f"Number of time points: {len(data['marginals'])}")
print(f"Training times: {data['train_times']}")
print(f"Holdout times (for evaluation): {data['holdout_times']}")
print(f"Data dimension: {data['dim']}")
for t, m in data['marginals'].items():
    print(f"  Time {t}: {len(m)} cells")

## 2. Create DataLoaders

In [ ]:
train_loader, val_loader = create_eb_dataloaders(
    marginals=data['marginals_list'],
    train_times=data['train_times'],
    batch_size=128,
    val_split=0.1,
    ot_alignments=data['ot_alignments'],
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 3. Define Potentials and Model

We define potentials at intermediate time points to enforce marginal constraints.

In [ ]:
# Training times mapped to [0, 1]
train_times = data['train_times']
tks = [(t - min(train_times)) / (max(train_times) - min(train_times)) 
       for t in train_times[1:-1]]  # Intermediate times only

print(f"Intermediate time points (tks): {tks}")

# Create potentials
potentials = OrderedDict()
for tk in tks:
    potentials[tk] = W2InfPotential(
        tk=tk,
        strength=500.0,
        lambda_type='gaussian',
        width=0.2,
    )

# Create model
model = OTPFM(
    d=data['dim'],
    tks=tks,
    potentials=potentials,
    flownet_args={
        'hidden_dim': 256,
        'num_hidden_layers': 3,
    }
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training

We use the EBTrainer for training with automatic evaluation.

In [ ]:
# Set up trainer
trainer = EBTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=Path("runs/singlecell_demo"),
    lr=1e-3,
    epochs=50,  # Increase for better results
    otp_alpha_type='sigmoid',
    otp_alpha_slope=6.0,
    potentials=potentials,
    marginals=data['marginals'],
    train_times=data['train_times'],
    holdout_times=data['holdout_times'],
    scaler=data['scaler'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print(f"Training on: {trainer.device}")

In [ ]:
# Train the model
trainer.train()

## 5. Evaluate and Visualize

In [ ]:
# Plot losses
trainer.plot_losses(show=True)

In [ ]:
# Sample trajectories for visualization
model.eval()
source_time = min(data['train_times'])
x0 = data['marginals'][source_time][:100].to(trainer.device)

with torch.no_grad():
    trajectories, t_eval = model.sample(x0, n_steps=50, ema=True)

print(f"Trajectory shape: {trajectories.shape}")

# Visualize trajectories in PCA space using built-in plotting
plot_pca_trajectories(
    trajectories=trajectories,
    time_points=t_eval.numpy(),
    ground_truth_marginals=data['marginals'],
    plot_times=data['train_times'],
    pcs=(0, 1),
    num_trajectories=100,
    title="EB Single-Cell Trajectories",
    show=True,
)

## Next Steps

To reproduce paper results, run:
```bash
python experiments/train.py --dataset singlecell --potential {W2, W2Inf, KL, MMD}
```

See `REPRODUCIBILITY.md` for complete instructions.